In [ ]:
#Q1
def clean_training_records(records):
    rejected_indices = []
    cleaned = []
    for i in records:
        ln_id = str(i["learner_id"]).strip()
        ln_score = i["score"]
        if ln_id == "":
            rejected_indices.append(records.index(i))
        elif type(ln_score) == bool or type(ln_score) == type(None) or type(ln_score) == list or type(ln_score) == dict or type(ln_score) == tuple or type(ln_score) == set or type(ln_score) == frozenset or type(ln_score) == complex:
            rejected_indices.append(records.index(i))
        elif float(ln_score) > 100 or float(ln_score) < 0:
            rejected_indices.append(records.index(i))
        else:
            cleaned.append(i)
            if float(ln_score) >= 60:
                i["eligible"] = True
            elif float(ln_score) < 60:
                i["eligible"] = False
    print(cleaned)
    print(rejected_indices)

records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]
clean_training_records(records)


#Test case1:
records = [
    {"learner_id": 0 , "score": "82.5"},
    {"learner_id": "L02", "score": 60},
]
clean_training_records(records)

#Test case2:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
]
clean_training_records(records)


In [ ]:
#Q2
def prepare_messages(payloads, min_length=3):
    if type(min_length) is bool or not isinstance(min_length, int) or min_length <= 0:
        raise ValueError("min_length must be a positive integer excluding Booleans")

    token_lists = []
    rejected_indices = []
    
    for i, payload in enumerate(payloads):
        if isinstance(payload, str):
            text = payload
        elif isinstance(payload, (bytes, bytearray)):
            try:
                # bytearray decode doesn't modify the object
                text = payload.decode('utf-8')
            except UnicodeDecodeError:
                rejected_indices.append(i)
                continue
        else:
            rejected_indices.append(i)
            continue
            
        # Lowercase, split, strip punctuation
        cleaned = [word.strip('.,!?;:') for word in text.lower().split()]
        
        # Comprehension to keep tokens of at least min_length
        filtered_tokens = [word for word in cleaned if len(word) >= min_length]
        
        token_lists.append(filtered_tokens)
        
    # Build vocabulary using a set comprehension to gather unique tokens
    vocabulary = sorted({word for token_list in token_lists for word in token_list})
    
    return {
        "token_lists": token_lists,
        "vocabulary": vocabulary,
        "rejected_indices": rejected_indices,
    }

if __name__ == "__main__":
    import pprint
    payloads = [
        "  AI, helps TEAMS! ",
        b"Build reliable agents.",
        bytearray(b"AI helps."),
        None,
        b"\xff",
    ]
    print("=== Sample Output ===")
    pprint.pprint(prepare_messages(payloads, min_length=3), sort_dicts=False)
    
    print("\n=== Additional Test Cases ===")
    print("Test Case 1: Empty inputs")
    pprint.pprint(prepare_messages(["", b"   "], min_length=3))
    
    print("\nTest Case 2: Internal punctuation preservation")
    pprint.pprint(prepare_messages(["agent-based architecture is reliable!"], min_length=4))
    
    print("\nTest Case 3: Rejecting ints and filtering lengths carefully")
    pprint.pprint(prepare_messages(["a ab abc abcd", 123, bytearray(b"ab abc")], min_length=3))


In [ ]:
#Q3
def summarise_predictions(results):
    validres = []
    misclassified = []
    rejected_indices = []
    cor_pred_count = 0
    by_actual = {"billing" : {"total": 0, "correct": 0}, "technical" : {"total": 0, "correct": 0}, "account" : {"total": 0, "correct": 0}}

    for i in results:
        if not i.get("id") or not i.get("actual") or not i.get("predicted"):
            rejected_indices.append(results.index(i))
            continue

        ln_id = str(i["id"]).strip()
        ln_actual = str(i["actual"]).strip().lower()
        ln_predicted = str(i["predicted"]).strip().lower()
        
        if ln_id == "":
            rejected_indices.append(results.index(i))

        validres.append(i)

        if ln_actual == ln_predicted:
            cor_pred_count += 1
            by_actual[ln_actual]["correct"] += 1
        else:
            misclassified.append(ln_id)

        by_actual[ln_actual]["total"] += 1

    accuracy = cor_pred_count / len(validres) if validres else 0.0


    print("Valid_count :", len(validres))
    print("Correct_pred_count :", cor_pred_count)
    print("Accuracy :", round(accuracy, 4))
    print(by_actual)
    print("Misclassified :", misclassified)
    print("Rejected_Indices :", rejected_indices)


results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
    {"id": "T5", "actual": None, "predicted": "billing"}
]
summarise_predictions(results)

results = [{"id": 1, "actual": "Billing", "predicted": "billing"}, 
           {"id": 2, "actual": "Account", "predicted": "technical"}]
summarise_predictions(results)

results = [{"id": 1, "actual": None, "predicted": "Billing"}, 
           {"id": 2, "actual": "Technical", "predicted": None}]
summarise_predictions(results)


In [ ]:
#Q4
import pprint



def audit_dataset_ids(train_ids, validation_ids):
    if train_ids is None:
        train_ids = []
    if validation_ids is None:
        validation_ids = []

    def process_ids(id_list):
        invalid_indices = []
        cleaned_ids = []
        seen = set()
        dup = []

        for idx, item in enumerate(id_list):
            if isinstance(item, str):
                cleaned = item.strip()
                if cleaned != "":
                    cleaned_ids.append(cleaned)
                    if cleaned in seen:
                        if cleaned not in dup:
                            dup.append(cleaned)
                    else:
                        seen.add(cleaned)
                else:
                    invalid_indices.append(idx)
            else:
                invalid_indices.append(idx)

        return cleaned_ids, dup, invalid_indices

    train_cleaned, train_dup, invalid_train_indices = process_ids(train_ids)
    val_cleaned, val_dup, invalid_val_indices = process_ids(validation_ids)

    train_set = set(train_cleaned)
    val_set = set(val_cleaned)

    overlap = sorted(list(train_set & val_set))
    train_only = sorted(list(train_set - val_set))
    val_only = sorted(list(val_set - train_set))

    result = {
        "train_duplicates": train_dup,
        "validation_duplicates": val_dup,
        "overlap": overlap,
        "train_only": train_only,
        "validation_only": val_only,
        "invalid_train_indices": invalid_train_indices,
        "invalid_validation_indices": invalid_val_indices,
        "snapshots": {
            "train": frozenset(train_set),
            "validation": frozenset(val_set),
        },
    }

    return result



# Sample Test Case
train_ids = ["R3", "R1", " R2 ", "R1", None]
validation_ids = ["R2", "R4", "R4", " "]
res = audit_dataset_ids(train_ids, validation_ids)
pprint.pprint(res, sort_dicts=False)
print()

# Test Case1
train_ids = ["R1", "R3", "r2", "R2", None]
validation_ids = ["R1", "r3", "r2", "R2", " "]
res = audit_dataset_ids(train_ids, validation_ids)
pprint.pprint(res, sort_dicts=False)
print()

# Test Case2
train_ids = []
validation_ids = None
res = audit_dataset_ids(train_ids, validation_ids)
pprint.pprint(res, sort_dicts=False)

In [ ]:
#Q5
import pprint

def update_annotation_queue(queue, incoming, completed, priority=None, batch_size=3):
    if type(batch_size) is bool or not isinstance(batch_size, int) or batch_size < 0:
        raise ValueError("batch_size must be a non-negative integer excluding Booleans")

    combined = queue + incoming
    
    q = []
    for d in combined:
        if d not in q and d not in completed:
            q.append(d)
            
    if priority in q:
        q.remove(priority)
        q.insert(0, priority)
        
    next_batch = q[:batch_size]
    
    last_item = q[-1] if len(q) > 0 else None
    
    positions = []
    for i in range(len(q)):
        positions.append((i + 1, q[i]))
        
    return {
        "queue": q,
        "next_batch": next_batch,
        "last_item": last_item,
        "positions": positions
    }

queue = ["D3", "D1", "D3", "D2"]
incoming = ["D4", "D2", "D5"]
completed = ["D1", "D9"]

res = update_annotation_queue(
    queue, incoming, completed,
    priority="D5", batch_size=3
)
pprint.pprint(res, sort_dicts=False)

t1_res = update_annotation_queue([], [], [])
pprint.pprint(t1_res, sort_dicts=False)

t2_res = update_annotation_queue(["A", "B"], ["C"], ["A"], priority="Z", batch_size=0)
pprint.pprint(t2_res, sort_dicts=False)






In [ ]:
#Q6
import copy
import pprint
from typing import Dict, Any, Optional

def build_experiment_config(base: Dict[str, Any], *, run_name: str,
                            parameter_updates: Optional[Dict[str, Any]] = None, **metadata: Any) -> Dict[str, Any]:
    # Trim the run_name
    rn = run_name.strip()
    if not rn:
        raise ValueError("run_name must be a non-empty string after trimming")
        
    # Validate parameter_updates
    if parameter_updates is not None and not isinstance(parameter_updates, dict):
        raise TypeError("parameter_updates must be a dictionary or None")
        
    # Create an independent copy
    c = copy.deepcopy(base)
    
    # Add run_name
    c["run_name"] = rn
    
    # Apply parameter updates safely
    if parameter_updates:
        for k, v in parameter_updates.items():
            if k not in c["parameters"]:
                raise KeyError(f"Unknown parameter: {k}")
            c["parameters"][k] = v
            
    # Merge any additional metadata
    if metadata:
        c["metadata"].update(metadata)
        
    return c

# ==========================================
# Testing & Deliverables
# ==========================================

base = {
    "model": "ticket_classifier",
    "parameters": {"threshold": 0.5, "max_tokens": 128},
    "metadata": {"owner": "platform", "stage": "baseline"},
}

print("=== Sample Output ===")
res = build_experiment_config(
    base, run_name="trial_02",
    parameter_updates={"threshold": 0.7},
    owner="nlp_team", seed=42,
)
pprint.pprint(res, sort_dicts=False)

print("\n=== Additional Test Cases ===")

print("\nTest Case 1: Independence test (Mutating returned config does not affect base)")
indep_res = build_experiment_config(base, run_name="  trial_03  ", parameter_updates=None)
indep_res["parameters"]["threshold"] = 0.99
indep_res["metadata"]["owner"] = "hacker"

print("Returned config parameters:", indep_res["parameters"])
print("Original base parameters:", base["parameters"])
print("Is original base intact?", base["parameters"]["threshold"] == 0.5 and base["metadata"]["owner"] == "platform")

print("\nTest Case 2: KeyError on unknown parameter")
try:
    build_experiment_config(base, run_name="fail_test", parameter_updates={"learning_rate": 0.01})
except KeyError as e:
    print("Caught expected KeyError:", e)
    
print("\nTest Case 3: ValueError on empty run_name")
try:
    build_experiment_config(base, run_name="   \t \n ")
except ValueError as e:
    print("Caught expected ValueError:", e)
    
print("\nTest Case 4: TypeError on invalid parameter_updates")
try:
    build_experiment_config(base, run_name="type_test", parameter_updates=["invalid", "list"])
except TypeError as e:
    print("Caught expected TypeError:", e)


In [ ]:
#Q7
import math

def aggregate_scores(*batches, top_n=3):
    if type(top_n) is bool or not isinstance(top_n, int) or top_n < 0:
        raise ValueError("top_n must be a non-negative integer excluding Booleans")
        
    for batch in batches:
        if not isinstance(batch, (list, tuple)):
            raise TypeError("Each positional batch must be a list or tuple")

    valid_entries = []
    rejected_positions = []
    
    for b_idx, batch in enumerate(batches):
        for i_idx, item in enumerate(batch):
            # Exclude booleans, strings, and accept finite ints/floats
            if isinstance(item, (int, float)) and type(item) is not bool:
                if math.isfinite(item) and 0 <= item <= 100:
                    valid_entries.append((b_idx, i_idx, float(item)))
                    continue
            rejected_positions.append((b_idx, i_idx))
            
    valid_count = len(valid_entries)
    if valid_count > 0:
        # Using a generator expression to calculate the total sum
        total = sum(score for _, _, score in valid_entries)
        mean = round(total / valid_count, 2)
    else:
        mean = None
        
    # Sort descending by score, then ascending by batch_idx, then item_idx
    valid_entries.sort(key=lambda x: (-x[2], x[0], x[1]))
    top_scores = valid_entries[:top_n]
    
    return {
        "valid_count": valid_count,
        "mean": mean,
        "top_scores": top_scores,
        "rejected_positions": rejected_positions,
    }

if __name__ == "__main__":
    import pprint
    batch_a = [80, 90, None]
    batch_b = (90, 70, True)
    batch_c = []
    print("=== Sample Output ===")
    pprint.pprint(aggregate_scores(batch_a, batch_b, batch_c, top_n=3), sort_dicts=False)
    
    print("\n=== Additional Test Cases ===")
    print("Test Case 1: TypeError on invalid batch type")
    try:
        aggregate_scores([10, 20], "invalid_batch_string")
    except TypeError as e:
        print("Caught expected TypeError:", e)
        
    print("\nTest Case 2: Zero scores and empty outputs")
    pprint.pprint(aggregate_scores([None, True, "100"], top_n=5))
    
    print("\nTest Case 3: Rejecting out of range, nan, and string numbers")
    pprint.pprint(aggregate_scores([101, -1, float('nan'), "50"], top_n=2))


In [ ]:
#Q8
def summarise_tool_calls(records):
    # Dictionary to keep only the final record per call_id
    final_calls = {}
    for r in records:
        final_calls[r['call_id']] = r
        
    unique_tools = set(r['tool'] for r in final_calls.values())
    
    # Dictionary comprehension for initialising the report
    by_tool = {
        tool: {'calls': 0, 'successes': 0, 'failures': 0, 'total_duration_ms': 0}
        for tool in unique_tools
    }
    
    failed_call_ids = []
    
    for r in final_calls.values():
        tool = r['tool']
        by_tool[tool]['calls'] += 1
        by_tool[tool]['total_duration_ms'] += r['duration_ms']
        
        if r['status'] == 'error':
            by_tool[tool]['failures'] += 1
            failed_call_ids.append(r['call_id'])
        elif r['status'] == 'ok':
            by_tool[tool]['successes'] += 1
            
    # Sort lexicographically
    failed_call_ids.sort()
    
    # Ranking based on -failures, -calls, and alphabetically on tool
    tool_ranking = sorted(
        unique_tools, 
        key=lambda t: (-by_tool[t]['failures'], -by_tool[t]['calls'], t)
    )
    
    return {
        'unique_calls': len(final_calls),
        'by_tool': by_tool,
        'failed_call_ids': failed_call_ids,
        'tool_ranking': tool_ranking
    }

if __name__ == "__main__":
    import pprint
    records = [
        {"call_id": "C1", "tool": "search", "status": "error", "duration_ms": 120},
        {"call_id": "C2", "tool": "calculator", "status": "ok", "duration_ms": 10},
        {"call_id": "C1", "tool": "search", "status": "ok", "duration_ms": 80},
        {"call_id": "C3", "tool": "search", "status": "error", "duration_ms": 40},
        {"call_id": "C4", "tool": "retriever", "status": "error", "duration_ms": 50},
    ]
    print("=== Sample Output ===")
    pprint.pprint(summarise_tool_calls(records), sort_dicts=False)
    
    print("\n=== Additional Test Cases ===")
    
    print("Test Case 1: Empty input")
    pprint.pprint(summarise_tool_calls([]), sort_dicts=False)
    
    print("\nTest Case 2: Replacing status and tool completely")
    change_records = [
        {"call_id": "X1", "tool": "search", "status": "error", "duration_ms": 100},
        {"call_id": "X1", "tool": "calculator", "status": "ok", "duration_ms": 20},
    ]
    pprint.pprint(summarise_tool_calls(change_records), sort_dicts=False)
    
    print("\nTest Case 3: Tool-ranking tie-breaking rules")
    tie_records = [
        {"call_id": "T1", "tool": "zeta", "status": "error", "duration_ms": 10},
        {"call_id": "T2", "tool": "alpha", "status": "error", "duration_ms": 10},
        {"call_id": "T3", "tool": "beta", "status": "error", "duration_ms": 10},
    ]
    print("Tie Ranking Output:", summarise_tool_calls(tie_records)["tool_ranking"])


In [ ]:
#Q9
import functools
from typing import List

def validate_text_batch(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Determine the texts argument (positional vs keyword)
        if len(args) > 0:
            texts = args[0]
        elif 'texts' in kwargs:
            texts = kwargs['texts']
        else:
            raise TypeError("The 'texts' argument is missing.")
            
        # Enforce list type
        if not isinstance(texts, list):
            raise TypeError("Input 'texts' must be a list.")
            
        # First Pass: Check for non-string items (Raises TypeError)
        for t in texts:
            if not isinstance(t, str):
                raise TypeError("Every item in the batch must be a string.")
                
        # Second Pass: Check for blank strings (Raises ValueError)
        for t in texts:
            if not t.strip():
                raise ValueError("Strings cannot be empty or only whitespace.")
                
        return func(*args, **kwargs)
    return wrapper

@validate_text_batch
def text_lengths(texts: List[str], *, strip: bool = True) -> List[int]:
    '''Calculate character counts for a batch of text.'''
    if strip:
        return [len(t.strip()) for t in texts]
    else:
        return [len(t) for t in texts]


if __name__ == "__main__":
    texts = ["  agent ", "ML", "data"]
    
    print("=== Sample Output ===")
    print("# First call:")
    print(text_lengths(texts))
    print("\n# Second call:")
    print(text_lengths(texts=texts, strip=False))
    
    print("\n=== Additional Test Cases ===")
    print("Test Case 1: Empty list is valid")
    print("Output:", text_lengths([]))
    
    print("\nTest Case 2: TypeError precedes ValueError if both exist")
    try:
        text_lengths(["   ", 123])
        print("Execution successful!")
    except Exception as e:
        print(f"Caught Exception: {type(e).__name__} -> {e}")
        
    print("\nTest Case 3: Invalid input prevents execution")
    try:
        text_lengths(["valid", "   "])
        print("Execution successful! (This shouldn't print)")
    except ValueError as e:
        print(f"Caught Exception: {type(e).__name__} -> {e}")
        print("Function did not execute because validation blocked it.")


In [ ]:
#10
import functools
import math

def track_calls(history):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # 1-based call numbering across the shared history
            call_number = len(history) + 1
            record = {
                "call_number": call_number,
                "function": func.__name__,
                "status": "success",
                "error_type": None
            }
            try:
                result = func(*args, **kwargs)
                history.append(record)
                return result
            except Exception as e:
                record["status"] = "error"
                record["error_type"] = type(e).__name__
                history.append(record)
                raise
        return wrapper
    return decorator

history = []

@track_calls(history)
def average_score(scores):
    if not isinstance(scores, list):
        raise TypeError("scores must be a list")
    if not scores:
        raise ValueError("scores list cannot be empty")
    for s in scores:
        if type(s) is bool or not isinstance(s, (int, float)) or not math.isfinite(s):
            raise TypeError("scores must contain finite integers or floats")
    return sum(scores) / len(scores)

@track_calls(history)
def format_run_name(prefix, *, number):
    if not isinstance(prefix, str) or not prefix:
        raise ValueError("prefix must be a non-empty string")
    if type(number) is bool or not isinstance(number, int) or number < 0:
        raise ValueError("number must be a non-negative integer")
    return f"{prefix}_{number:02d}"

if __name__ == "__main__":
    import pprint
    
    print("=== Sample Output ===")
    print("# First call returns:")
    print(average_score([60, 80]))
    
    print("# Second call returns:")
    print(format_run_name("eval", number=3))
    
    print("# Third call raises ValueError.")
    try:
        average_score([])
    except ValueError:
        pass
        
    print("\n# History after all three attempted calls:")
    pprint.pprint(history, sort_dicts=False)
    
    print("\n=== Additional Test Cases ===")
    print("Test Case 1: Returning None explicitly records as success")
    
    @track_calls(history)
    def returns_none():
        return None
        
    returns_none()
    print("Last record:", history[-1])
    
    print("\nTest Case 2: TypeError propagates seamlessly")
    try:
        format_run_name("eval", number=False) # boolean raises ValueError
    except ValueError as e:
        print(f"Caught Exception: ValueError -> {e}")
    print("Last record:", history[-1])
    
    print("\nTest Case 3: Independent histories")
    history2 = []
    
    @track_calls(history2)
    def independent_func():
        return "independent"
        
    independent_func()
    print("History 2:", history2)
    print("Original History Length:", len(history))
